# Bibliotecas

In [1]:
#!pip install -r requirements_all.txt

In [2]:
#!pip install datasets

In [3]:
import numpy as np
import os
from datetime import datetime

import datasets
import evaluate

import torch
import torch.nn as nn

from transformers import Trainer, TrainerCallback, TrainingArguments, EarlyStoppingCallback
from transformers import AutoTokenizer, BertConfig, BertModel, BertPreTrainedModel

/home/guilhermelima/msc/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Carregar os conjunto de dados pré-processado

In [4]:
#MAX_LENGTH = 512
NUM_TRAIN_EPOCHS = 10

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

RESULTS_DIRECTORY = './results/experiment_{}'.format(timestamp)

LOGGING_DIRECTORY = './logs/experiement_{}'.format(timestamp)

RESULTS_DIRECTORY, LOGGING_DIRECTORY

('./results/experiment_2026-06-15_09-26-18',
 './logs/experiement_2026-06-15_09-26-18')

In [5]:
#XPOS_LABELS = ['DET', 'NOUN', 'ADP', 'ADJ', 'VERB', 'CONJ', 'ADV', 'AUX', '.', 'PNOUN', 'NUM', 'PRON', 'PRT', 'ADPPRON', 'X']

#DEPREL_LABELS = ['det', 'attr', 'adpmod', 'amod', 'adpobj', 'ROOT', 'mark', 'nsubj', 'advmod', 'aux', 'adp', 'csubj', 'cc', 'conj', 'p', 'compmod', 'appos', 'acomp', 'num', 'nsubjpass', 'auxpass', 'dobj', 'poss', 'partmod', 'xcomp', 'ccomp', 'adpcomp', 'advcl', 'rcmod', 'nmod', 'neg', 'mwe', 'dep', 'parataxis', 'iobj', 'prt', 'infmod', 'csubjpass']

#UPOS_LABELS = ['DET', 'NOUN', 'ADP', 'ADJ', 'VERB', 'CONJ', 'ADV', '.', 'NUM', 'PRON', 'PRT', 'X']

UPOS_LABELS = ['DET', 'NOUN', 'VERB', 'PUNCT', 'SCONJ', 'ADP', 'ADJ', 'CCONJ', 'ADV', 'PROPN', 'AUX', 'NUM', 'PRON', 'SYM', 'X', 'INTJ']
DEPREL_LABELS = ['det', 'nsubj', 'root', 'obj', 'xcomp', 'punct', 'mark', 'advcl', 'case', 'obl', 'amod', 'conj', 'cc', 'nmod', 'advmod', 'flat:name', 'ccomp', 'cop', 'acl', 'nummod', 'acl:relcl', 'ccomp:speech', 'parataxis', 'csubj', 'aux:pass', 'appos', 'fixed', 'nsubj:pass', 'aux', 'nsubj:outer', 'obl:agent', 'expl:impers', 'expl', 'discourse', 'orphan', 'dislocated', 'flat', 'flat:foreign', 'iobj', 'vocative', 'csubj:outer', 'list', 'reparandum', 'csubj:pass']

DEPREL_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(DEPREL_LABELS)
}

IDX_TO_DEPREL_LABELS = {
    i: j
    for j, i in DEPREL_LABELS_TO_IDX.items()
}


UPOS_LABELS_TO_IDX = {
    i : idx
    for idx, i in enumerate(UPOS_LABELS)
}

IDX_TO_UPOS_LABELS = {
    i: j
    for j, i in UPOS_LABELS_TO_IDX.items()
}


#MAX_SEQUENCE_LENGTH = 512#
#PRETRAINED_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
#PRETRAINED_MODEL_NAME = "google-bert/bert-base-multilingual-cased"
NUM_TRAIN_EPOCHS = 10
PRETRAINED_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

In [6]:
# initializing Config and Tokenizer
BERT_CONFIG = BertConfig.from_pretrained(PRETRAINED_MODEL_NAME)
TOKENIZER = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased", use_fast=True) #.is_fast

In [7]:
BERT_CONFIG = {
    "vocab_size": 29794,
    "hidden_size": 768,
    "num_hidden_layers": 12,
    "num_attention_heads": 12,
    "intermediate_size": 3072,
    "hidden_act": "gelu",
    "max_position_embeddings": 512,
    "type_vocab_size": 2,
    "initializer_range": 0.02,
    "layer_norm_eps": 1e-12,
    "pad_token_id": 0,
    "attention_probs_dropout_prob": 0.1,
    "hidden_dropout_prob": 0.1,
}

In [8]:
import ast
import pandas as pd
import datasets
from datasets import Dataset, DatasetDict

def load_csv_as_hf_dataset(filepath):
    df = pd.read_csv(filepath)
    records = []
    for _, row in df.iterrows():
        records.append({
            'tokens': ast.literal_eval(row['tokens']),
            'upos': ast.literal_eval(row['upos']),
            'deprel': ast.literal_eval(row['deprel']),
            'head_tags': ast.literal_eval(str(row['head_tags'])),
            'deprel_tags': ast.literal_eval(str(row['deprel_tags'])),
            'upos_tags': ast.literal_eval(str(row['upos_tags'])),
        })
    return Dataset.from_list(records)

data = DatasetDict({
    'train': load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/train_outxpos.csv'),
    'val':   load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/val_outxpos.csv'),
    'test':  load_csv_as_hf_dataset('/home/guilhermelima/msc/data_dois/test_outxpos.csv'),
})
data

DatasetDict({
    train: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 5893
    })
    val: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 842
    })
    test: Dataset({
        features: ['tokens', 'upos', 'deprel', 'head_tags', 'deprel_tags', 'upos_tags'],
        num_rows: 1683
    })
})

# Modelo

In [9]:
'''
class Biaffine(nn.Module):
    def __init__(self, in_features, out_features=1, bias_x=True, bias_y=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bias_x = bias_x
        self.bias_y = bias_y

        self.weight = nn.Parameter(torch.Tensor(out_features, in_features + int(bias_x), in_features + int(bias_y)))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, y):
        # x, y: [B, L, H]
        if self.bias_x:
            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)  # [B, L, H+1]
        if self.bias_y:
            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)  # [B, L, H+1]
        # cálculo biaffine
        # weight: [O, H+1, H+1]
        # resultado: [B, O, L, L]
        logits = torch.einsum('bxi,oij,byj->boxy', x, self.weight, y)
        return logits.squeeze(1) if self.out_features == 1 else logits
'''

"\nclass Biaffine(nn.Module):\n    def __init__(self, in_features, out_features=1, bias_x=True, bias_y=True):\n        super().__init__()\n        self.in_features = in_features\n        self.out_features = out_features\n        self.bias_x = bias_x\n        self.bias_y = bias_y\n\n        self.weight = nn.Parameter(torch.Tensor(out_features, in_features + int(bias_x), in_features + int(bias_y)))\n        nn.init.xavier_uniform_(self.weight)\n\n    def forward(self, x, y):\n        # x, y: [B, L, H]\n        if self.bias_x:\n            x = torch.cat([x, x.new_ones(*x.shape[:-1], 1)], dim=-1)  # [B, L, H+1]\n        if self.bias_y:\n            y = torch.cat([y, y.new_ones(*y.shape[:-1], 1)], dim=-1)  # [B, L, H+1]\n        # cálculo biaffine\n        # weight: [O, H+1, H+1]\n        # resultado: [B, O, L, L]\n        logits = torch.einsum('bxi,oij,byj->boxy', x, self.weight, y)\n        return logits.squeeze(1) if self.out_features == 1 else logits\n"

In [10]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer


In [11]:
from transformers import PreTrainedModel, AutoModel
import torch.nn as nn
import torch

#DECODER

class MultiTaskSentencePrediction(PreTrainedModel):
    # Permite carregar QUALQUER config de modelo decoder
    config_class = None

    def __init__(self, config, num_deprel_labels, num_upos_labels, num_head_labels=100):
        super().__init__(config)

        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels
        self.num_head_labels = num_head_labels


        self.model = AutoModel.from_config(config)

        hidden = config.hidden_size

        self.deprel_classifier = nn.Linear(hidden, num_deprel_labels)
        self.upos_classifier   = nn.Linear(hidden, num_upos_labels)
        self.head_classifier   = nn.Linear(hidden, num_head_labels)


        dropout_prob = getattr(config, "hidden_dropout", None) \
                       or getattr(config, "hidden_dropout_prob", None) \
                       or getattr(config, "classifier_dropout", 0.1) \
                       or 0.1

        self.dropout = nn.Dropout(dropout_prob)

        self.post_init()

    def forward(
        self,
        input_ids,
        attention_mask=None,
        position_ids=None,
        deprel_label=None,
        upos_label=None,
        head_label=None
    ):

        model_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        # decoders geralmente aceitam position_ids
        if position_ids is not None and "position_ids" in self.model.forward.__code__.co_varnames:
            model_kwargs["position_ids"] = position_ids

        outputs = self.model(**model_kwargs)


        h = self.dropout(outputs.last_hidden_state)

        logits_deprel = self.deprel_classifier(h)
        logits_upos   = self.upos_classifier(h)
        logits_head   = self.head_classifier(h)

        loss = None
        if (
            deprel_label is not None
            and upos_label is not None
            and head_label is not None
        ):

                loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
                loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)
                loss_fct3 = nn.CrossEntropyLoss(ignore_index=-100)


                loss = loss_fct1(
                logits_deprel.view(-1, self.num_deprel_labels),  # [B * L, num_xpos_labels]
                deprel_label.view(-1)                          # [B * L]
            ) + loss_fct2(
                logits_upos.view(-1, self.num_upos_labels),
                upos_label.view(-1)
            ) + loss_fct3(
                logits_head.view(-1, self.num_head_labels),
                head_label.view(-1)
            )

        if loss is not None:
            return loss, logits_deprel, logits_upos, logits_head

        return logits_deprel, logits_upos, logits_head


In [12]:
'''import torch
import torch.nn as nn

class MultiTaskDecoder(nn.Module):
    def __init__(self, hidden_size, num_deprel_labels, num_upos_labels, num_head_labels=100, num_layers=6, num_heads=8, dropout_prob=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels
        self.num_head_labels = num_head_labels

        # Decoder Transformer
        decoder_layer = nn.TransformerDecoderLayer(d_model=hidden_size, nhead=num_heads, dropout=dropout_prob)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout_prob)

        # Cabeças de classificação
        self.deprel_classifier = nn.Linear(hidden_size, num_deprel_labels)
        self.upos_classifier = nn.Linear(hidden_size, num_upos_labels)
        self.head_classifier = nn.Linear(hidden_size, num_head_labels)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,   # precisa aceitar mesmo que você não use
        token_type_ids=None,   # pode ignorar se não usar
        deprel_label=None,
        upos_label=None,
        head_label=None
    ):
        # input_ids -> embeddings simuladas ou reais
        # se for decoder puro sem embedding layer, transforme input_ids em embeddings
        # exemplo: embeddings = self.embedding(input_ids)

        # Simulando sequence_output: [batch_size, seq_len, hidden_size]
        sequence_output = torch.randn(input_ids.size(0), input_ids.size(1), self.hidden_size, device=input_ids.device)

        sequence_output = self.dropout(sequence_output)

        logits_deprel = self.deprel_classifier(sequence_output)
        logits_upos = self.upos_classifier(sequence_output)
        logits_head = self.head_classifier(sequence_output)

        loss = None
        if deprel_label is not None and upos_label is not None and head_label is not None:
            loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)
            loss_fct3 = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct1(
            logits_deprel.view(-1, self.num_deprel_labels),  # [B * L, num_xpos_labels]
            deprel_label.view(-1)                          # [B * L]
            ) + loss_fct2(
                logits_upos.view(-1, self.num_upos_labels),
                upos_label.view(-1)
            ) + loss_fct3(
                logits_head.view(-1, self.num_head_labels),
                head_label.view(-1)
            )

        return (loss, logits_deprel, logits_upos, logits_head) if loss is not None else (logits_deprel, logits_upos, logits_head)
'''

'import torch\nimport torch.nn as nn\n\nclass MultiTaskDecoder(nn.Module):\n    def __init__(self, hidden_size, num_deprel_labels, num_upos_labels, num_head_labels=100, num_layers=6, num_heads=8, dropout_prob=0.1):\n        super().__init__()\n        self.hidden_size = hidden_size\n        self.num_deprel_labels = num_deprel_labels\n        self.num_upos_labels = num_upos_labels\n        self.num_head_labels = num_head_labels\n\n        # Decoder Transformer\n        decoder_layer = nn.TransformerDecoderLayer(d_model=hidden_size, nhead=num_heads, dropout=dropout_prob)\n        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)\n        self.dropout = nn.Dropout(dropout_prob)\n\n        # Cabeças de classificação\n        self.deprel_classifier = nn.Linear(hidden_size, num_deprel_labels)\n        self.upos_classifier = nn.Linear(hidden_size, num_upos_labels)\n        self.head_classifier = nn.Linear(hidden_size, num_head_labels)\n\n    def forward(\n        self

In [13]:
# model definition
'''class MultiTaskSentencePredictionEncoderBiaffine(BertPreTrainedModel):
    def __init__(self, config, num_deprel_labels, num_upos_labels):
        super().__init__(config)
        
        self.num_deprel_labels = num_deprel_labels
        self.num_upos_labels = num_upos_labels
        
        self.bert = BertModel(config)
        
        # Classificadores simples para DepRel e UPOS
        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
        self.upos_classifier   = nn.Linear(config.hidden_size, num_upos_labels)
        
        # Biaffine para heads
        self.head_mlp = nn.Linear(config.hidden_size, config.hidden_size)
        self.dep_mlp  = nn.Linear(config.hidden_size, config.hidden_size)
        self.head_classifier = Biaffine(in_features=config.hidden_size, out_features=1)
        
        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        self.dropout = nn.Dropout(classifier_dropout)
        
        self.init_weights()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None,
                deprel_label=None, upos_label=None, head_label=None):
        
        outputs = self.bert(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        sequence_output = self.dropout(outputs[0])  # [B, L, H]
        
        # Classificação UPOS e DepRel
        logits_deprel = self.deprel_classifier(sequence_output)
        logits_upos   = self.upos_classifier(sequence_output)
        
        # Classificação de heads usando Biaffine
        head_repr = self.head_mlp(sequence_output)  # [B, L, H]
        dep_repr  = self.dep_mlp(sequence_output)   # [B, L, H]
        logits_head = self.head_classifier(head_repr, dep_repr).squeeze(1)  # [B, L, L]
        
        loss = None
        if deprel_label is not None and upos_label is not None and head_label is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            
            loss = loss_fct(
                logits_deprel.view(-1, self.num_deprel_labels),
                deprel_label.view(-1)
            ) + loss_fct(
                logits_upos.view(-1, self.num_upos_labels),
                upos_label.view(-1)
            ) + loss_fct(
                logits_head.view(-1, logits_head.size(-1)),  # [B*L, L]
                head_label.view(-1)
            )
        
        return (loss, logits_deprel, logits_upos, logits_head) if loss is not None else (logits_deprel, logits_upos, logits_head)'''


'class MultiTaskSentencePredictionEncoderBiaffine(BertPreTrainedModel):\n    def __init__(self, config, num_deprel_labels, num_upos_labels):\n        super().__init__(config)\n        \n        self.num_deprel_labels = num_deprel_labels\n        self.num_upos_labels = num_upos_labels\n        \n        self.bert = BertModel(config)\n        \n        # Classificadores simples para DepRel e UPOS\n        self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)\n        self.upos_classifier   = nn.Linear(config.hidden_size, num_upos_labels)\n        \n        # Biaffine para heads\n        self.head_mlp = nn.Linear(config.hidden_size, config.hidden_size)\n        self.dep_mlp  = nn.Linear(config.hidden_size, config.hidden_size)\n        self.head_classifier = Biaffine(in_features=config.hidden_size, out_features=1)\n        \n        classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob\n        self.dropo

In [14]:
# ENCONDER
class MultiTaskSentencePredictionEncoder(BertPreTrainedModel):
        _tied_weights_keys = []
        all_tied_weights_keys = {}
        def __init__(self, config, num_deprel_labels, num_upos_labels, num_head_labels=200):
            super().__init__(config)
            #self.num_xpos_labels = num_xpos_labels
            
            self.num_deprel_labels = num_deprel_labels
            self.num_upos_labels = num_upos_labels
            self.num_head_labels = num_head_labels
            
            if False:
                self.bert = BertModel(config)
            else:
                self.bert = AutoModel.from_config(config)

            #self.xpos_classifier = nn.Linear(config.hidden_size, num_xpos_labels)
            self.deprel_classifier = nn.Linear(config.hidden_size, num_deprel_labels)
            self.upos_classifier = nn.Linear(config.hidden_size, num_upos_labels)
            self.head_classifier = nn.Linear(config.hidden_size, num_head_labels)
            
            #self.head_classifier = Biaffine(in_features=config.hidden_size, out_features=1)

            classifier_dropout = config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
            
            self.dropout = nn.Dropout(classifier_dropout)
            self.init_weights()

        def forward(
                self, input_ids, attention_mask=None, token_type_ids=None, 
                deprel_label=None, upos_label=None , head_label=None
        ):
            outputs = self.bert(
                input_ids=input_ids, attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
            sequence_output = self.dropout(outputs[0])  # [batch_size, seq_len, hidden_size]
            """
            def debug_labels(name, labels, num_classes):
                    print(f"\n{name}")
                    print("min:", labels.min().item())
                    print("max:", labels.max().item())
                    print("unique:", torch.unique(labels))

                    invalid = (labels >= num_classes) | (labels < -100)
                    if invalid.any():
                        print("❌ VALORES INVÁLIDOS ENCONTRADOS!")

                            # dentro do forward
            debug_labels("deprel", deprel_label, self.num_deprel_labels)
            debug_labels("upos", upos_label, self.num_upos_labels)
            debug_labels("head", head_label, self.num_head_labels)"""
            
            # Classificação por token
            #logits_xpos = self.xpos_classifier(sequence_output)     # [batch_size, seq_len, num_xpos_labels]
            logits_deprel = self.deprel_classifier(sequence_output)
            logits_upos = self.upos_classifier(sequence_output)
            logits_head = self.head_classifier(sequence_output)

            loss = None
            if deprel_label is not None and upos_label is not None and head_label is not None:
                
                
                loss_fct1 = nn.CrossEntropyLoss(ignore_index=-100)
                loss_fct2 = nn.CrossEntropyLoss(ignore_index=-100)
                loss_fct3 = nn.CrossEntropyLoss(ignore_index=-100)
                #loss_fct4 = nn.CrossEntropyLoss(ignore_index=-100)

                loss = loss_fct1(
                logits_deprel.view(-1, self.num_deprel_labels),  # [B * L, num_xpos_labels]
                deprel_label.view(-1)                          # [B * L]
            ) + loss_fct2(
                logits_upos.view(-1, self.num_upos_labels),
                upos_label.view(-1)
            ) + loss_fct3(
                logits_head.view(-1, self.num_head_labels),
                head_label.view(-1)
            )

            return (loss, logits_deprel, logits_upos, logits_head) if loss is not None else (logits_deprel, logits_upos, logits_head)


# Tokenização

In [15]:
class POSDataset:

    def __init__(self, tokenizer_ckpt):
        #self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt, add_prefix_space=True)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_ckpt)
        
    def align_labels_with_tokens(self, labels, word_ids):
        new_labels = []
        current_word = None
        for word_id in word_ids:
            if word_id != current_word:
                # Start of a new word!
                current_word = word_id
                try:
                    label = -100 if word_id is None else labels[word_id]
                except:
                    label = -100
                new_labels.append(label)
            elif word_id is None:
                # Special token
                new_labels.append(-100)
            else:
                # Same word as previous token
                label = labels[word_id]
                # If the label is B-XXX we change it to I-XXX
                #if label % 2 == 1:
                #    label += 1
                new_labels.append(label)

        return new_labels
    def preprocess_function(self, examples):
        tokenized_inputs = self.tokenizer(
        examples["tokens"], truncation=True , padding="max_length" , is_split_into_words=True, max_length=512)

        
        '''all_labels_xpos = examples["xpos_tags"]
        new_labels_xpos = []
        for i, labels in enumerate(all_labels_xpos):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_xpos.append(self.align_labels_with_tokens(labels, word_ids))'''

        all_labels_deprel = examples["deprel_tags"]
        new_labels_deprel = []
        for i, labels in enumerate(all_labels_deprel):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_deprel.append(self.align_labels_with_tokens(labels, word_ids))
        
        all_labels_upos = examples["upos_tags"]
        new_labels_upos = []
        for i, labels in enumerate(all_labels_upos):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_upos.append(self.align_labels_with_tokens(labels, word_ids))

        all_labels_head = examples["head_tags"]
        new_labels_head = []
        for i, labels in enumerate(all_labels_head):
            word_ids = tokenized_inputs.word_ids(i)
            new_labels_head.append(self.align_labels_with_tokens(labels, word_ids))

        #tokenized_inputs["xpos_label"] = new_labels_xpos
        tokenized_inputs["deprel_label"] = new_labels_deprel
        tokenized_inputs["upos_label"] = new_labels_upos
        tokenized_inputs["head_label"] = new_labels_head
        return tokenized_inputs

    def create_data(self, train, test):

        tokenized_train_dataset = train.map(
            self.preprocess_function,
            batched=True,
            remove_columns=train.column_names
        )

        tokenized_test_dataset = test.map(
            self.preprocess_function,
            batched=True,
            remove_columns= test.column_names
        )

        return tokenized_train_dataset, tokenized_test_dataset

In [16]:
#nerdataset = POSDataset("neuralmind/bert-base-portuguese-cased")
#nerdataset = POSDataset("google-bert/bert-base-multilingual-cased")
nerdataset = POSDataset("amadeusai/modernJabuticaBERT-Base-1k")

In [17]:
train_data, valid_data = nerdataset.create_data(data['train'], data['val'])

Map:   0%|                                      | 0/5893 [00:00<?, ? examples/s]

Map:  17%|████▏                    | 1000/5893 [00:00<00:02, 2046.61 examples/s]

Map:  34%|████████▍                | 2000/5893 [00:00<00:01, 2985.93 examples/s]

Map:  51%|████████████▋            | 3000/5893 [00:00<00:00, 3422.35 examples/s]

Map:  68%|████████████████▉        | 4000/5893 [00:01<00:00, 3750.34 examples/s]

...

Map: 100%|█████████████████████████| 5893/5893 [00:01<00:00, 3925.80 examples/s]

Map: 100%|█████████████████████████| 5893/5893 [00:01<00:00, 3548.41 examples/s]

Map:   0%|                                       | 0/842 [00:00<?, ? examples/s]

Map: 100%|███████████████████████████| 842/842 [00:00<00:00, 2260.59 examples/s]

Map: 100%|███████████████████████████| 842/842 [00:00<00:00, 2233.13 examples/s]

In [18]:
train_data

Dataset({
    features: ['input_ids', 'attention_mask', 'deprel_label', 'upos_label', 'head_label'],
    num_rows: 5893
})

# Data Collator

In [19]:
def data_collator(batch, padding_token_id=TOKENIZER.pad_token_id):
    input_ids = [item["input_ids"] for item in batch]
    attention_masks = [item["attention_mask"] for item in batch]
    #xpos_label = [item["xpos_label"] for item in batch]
    deprel_label = [item["deprel_label"] for item in batch]
    upos_label = [item["upos_label"] for item in batch]
    head_label  = [item["head_label"] for item in batch]
    
    
    max_len = max(len(ids) for ids in input_ids)
    '''
    for i, labels in enumerate(head_label):  # ou head_label, deprel_label
        for j, label in enumerate(labels):
            if label != -100 and (label < 0 or label >= 75):
                print(f"Erro no exemplo {i}, posição {j}: label={label}")'''

    input_ids = torch.tensor([ids + [padding_token_id] * (max_len - len(ids)) for ids in input_ids])
    attention_masks = torch.tensor([masks + [0] * (max_len - len(masks)) for masks in attention_masks])
    #xpos_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in xpos_label])
    deprel_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in deprel_label])
    upos_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in upos_label])
    head_label = torch.tensor([labels + [-100] * (max_len - len(labels)) for labels in head_label])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        #"xpos_label": xpos_label,
        "deprel_label": deprel_label,
        "upos_label": upos_label,
        "head_label": head_label
    }

In [20]:
'''def data_collator(batch, padding_token_id=0):
    input_ids = [item["input_ids"] for item in batch]

    # Cria attention_mask se não existir
    if "attention_mask" in batch[0]:
        attention_masks = [item["attention_mask"] for item in batch]
    else:
        attention_masks = [[1] * len(ids) for ids in input_ids]  # tudo 1s

    deprel_label = [item["deprel_label"] for item in batch]
    upos_label = [item["upos_label"] for item in batch]
    head_label  = [item["head_label"] for item in batch]

    max_len = max(len(ids) for ids in input_ids)

    # Padding
    input_ids = torch.tensor([ids + [padding_token_id] * (max_len - len(ids)) for ids in input_ids])
    attention_masks = torch.tensor([mask + [0] * (max_len - len(mask)) for mask in attention_masks])
    deprel_label = torch.tensor([lbl + [-100] * (max_len - len(lbl)) for lbl in deprel_label])
    upos_label = torch.tensor([lbl + [-100] * (max_len - len(lbl)) for lbl in upos_label])
    head_label = torch.tensor([lbl + [-100] * (max_len - len(lbl)) for lbl in head_label])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "deprel_label": deprel_label,
        "upos_label": upos_label,
        "head_label": head_label
    }
'''

'def data_collator(batch, padding_token_id=0):\n    input_ids = [item["input_ids"] for item in batch]\n\n    # Cria attention_mask se não existir\n    if "attention_mask" in batch[0]:\n        attention_masks = [item["attention_mask"] for item in batch]\n    else:\n        attention_masks = [[1] * len(ids) for ids in input_ids]  # tudo 1s\n\n    deprel_label = [item["deprel_label"] for item in batch]\n    upos_label = [item["upos_label"] for item in batch]\n    head_label  = [item["head_label"] for item in batch]\n\n    max_len = max(len(ids) for ids in input_ids)\n\n    # Padding\n    input_ids = torch.tensor([ids + [padding_token_id] * (max_len - len(ids)) for ids in input_ids])\n    attention_masks = torch.tensor([mask + [0] * (max_len - len(mask)) for mask in attention_masks])\n    deprel_label = torch.tensor([lbl + [-100] * (max_len - len(lbl)) for lbl in deprel_label])\n    upos_label = torch.tensor([lbl + [-100] * (max_len - len(lbl)) for lbl in upos_label])\n    head_label = to

In [21]:
#config_encoder = BertConfig(**BERT_CONFIG)

# Criação do Modelo

In [22]:
#model = MultiTaskSentencePrediction.from_pretrained(
#    PRETRAINED_MODEL_NAME,
#    config=BERT_CONFIG,
#    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS), num_head_labels=100
#)
from transformers import AutoConfig


config = AutoConfig.from_pretrained(PRETRAINED_MODEL_NAME)

# ESSENCIAL: registrar a classe no config
#MultiTaskSentencePrediction.config_class = config.__class_

In [23]:
model = MultiTaskSentencePrediction.from_pretrained(
    PRETRAINED_MODEL_NAME,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
)
#model = model.to("cpu")
#torch.cuda.empty_cache()

Loading weights: 0it [00:00, ?it/s]

Loading weights: 0it [00:00, ?it/s]

[transformers] MultiTaskSentencePrediction LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                                            | Status     | 
---------------------------------------------------------------+------------+-
bert.encoder.layer.{0...11}.attention.self.query.bias          | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.output.LayerNorm.bias    | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.query.weight        | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.key.bias            | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.self.key.weight          | UNEXPECTED | 
bert.encoder.layer.{0...11}.output.dense.bias                  | UNEXPECTED | 
bert.encoder.layer.{0...11}.attention.output.dense.bias        | UNEXPECTED | 
bert.encoder.layer.{0...11}.output.LayerNorm.bias              | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight                     | UNEXPECTED | 
bert.encoder.layer.{0...11}.atte

In [24]:
model

MultiTaskSentencePrediction(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

In [25]:
config = AutoConfig.from_pretrained(PRETRAINED_MODEL_NAME)

In [26]:
config = AutoConfig.from_pretrained(PRETRAINED_MODEL_NAME)

model = MultiTaskSentencePredictionEncoder.from_pretrained(
    PRETRAINED_MODEL_NAME,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
)


Loading weights:   0%|                                  | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 30937.64it/s]


[transformers] MultiTaskSentencePredictionEncoder LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
upos_classifier.weight                     | MISSING    | 
head_classifier.weight                     | MISSING    | 
head_classifier.bias                       | MISSING    | 
deprel_classifier.weight                   | MISSING    | 
deprel_classifier.bias                     | MISSING    | 
upos_cla

In [27]:
'''HIDDEN_SIZE = 512  # exemplo, ajuste conforme sua arquitetura
NUM_LAYERS = 6
NUM_HEADS = 8
DROPOUT = 0.1

# Instancia o decoder multitarefa
model = MultiTaskDecoder(
    hidden_size=HIDDEN_SIZE,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS),
    num_head_labels=100,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout_prob=DROPOUT
)'''

'HIDDEN_SIZE = 512  # exemplo, ajuste conforme sua arquitetura\nNUM_LAYERS = 6\nNUM_HEADS = 8\nDROPOUT = 0.1\n\n# Instancia o decoder multitarefa\nmodel = MultiTaskDecoder(\n    hidden_size=HIDDEN_SIZE,\n    num_deprel_labels=len(DEPREL_LABELS),\n    num_upos_labels=len(UPOS_LABELS),\n    num_head_labels=100,\n    num_layers=NUM_LAYERS,\n    num_heads=NUM_HEADS,\n    dropout_prob=DROPOUT\n)'

In [28]:
model

MultiTaskSentencePredictionEncoder(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

# Compute Metrics Old

In [29]:
import numpy as np


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    deprel_logits, upos_logits, head_logits = logits
    deprel_labels, upos_labels, head_labels = labels

    # Argmax
    deprel_preds = np.argmax(deprel_logits, axis=-1)
    upos_preds   = np.argmax(upos_logits, axis=-1)
    head_preds   = np.argmax(head_logits, axis=-1)

    # Máscara válida: token com HEAD e DEPREL anotados
    valid_mask = (
        (head_labels != -100) &
        (deprel_labels != -100)
    )

    # Filtrar
    head_preds = head_preds[valid_mask]
    head_labels = head_labels[valid_mask]

    deprel_preds = deprel_preds[valid_mask]
    deprel_labels = deprel_labels[valid_mask]

    # UAS: HEAD correto
    uas = (head_preds == head_labels).mean()

    # LAS: HEAD + DEPREL corretos
    las = ((head_preds == head_labels) &
           (deprel_preds == deprel_labels)).mean()

    # Métricas auxiliares (opcional)
    upos_mask = upos_labels != -100
    upos_acc = (upos_preds[upos_mask] == upos_labels[upos_mask]).mean()

    return {
        "uas": float(uas),
        "las": float(las),
        "upos_accuracy": float(upos_acc),
    }



# Compute Metrics

In [30]:
"""import numpy as np
import json
import os

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


def compute_metrics(eval_pred, PRETRAINED_MODEL=None, FOLD=None, TRIAL=None):

    save_path = "epoch_predictions/predictions_results.json"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # ---------------- IDENTIDADE DO EXPERIMENTO ----------------
    model_name = PRETRAINED_MODEL.split("/")[-1] if PRETRAINED_MODEL else "unknown_model"
    run_id = f"{model_name}_fold{FOLD}_trial{TRIAL}"

    logits, labels = eval_pred
    deprel_logits, upos_logits, head_logits = logits
    deprel_labels, upos_labels, head_labels = labels

    # ---------------- PROBABILIDADES ----------------
    deprel_probs = softmax(deprel_logits, axis=-1)
    upos_probs   = softmax(upos_logits, axis=-1)
    head_probs   = softmax(head_logits, axis=-1)

    # ---------------- PREDIÇÕES ----------------
    deprel_preds = np.argmax(deprel_logits, axis=-1)
    upos_preds   = np.argmax(upos_logits, axis=-1)
    head_preds   = np.argmax(head_logits, axis=-1)

    # ---------------- MASK DEPENDENCY ----------------
    valid_mask = (
        (head_labels != -100) &
        (deprel_labels != -100)
    )

    head_preds_masked   = head_preds[valid_mask]
    head_labels_masked  = head_labels[valid_mask]
    head_probs_masked   = head_probs[valid_mask]

    deprel_preds_masked  = deprel_preds[valid_mask]
    deprel_labels_masked = deprel_labels[valid_mask]
    deprel_probs_masked  = deprel_probs[valid_mask]

    # ---------------- MÉTRICAS PRINCIPAIS ----------------
    uas = (head_preds_masked == head_labels_masked).mean()

    las = (
        (head_preds_masked == head_labels_masked) &
        (deprel_preds_masked == deprel_labels_masked)
    ).mean()

    # ---------------- UPOS ----------------
    upos_mask = upos_labels != -100

    upos_preds_masked  = upos_preds[upos_mask]
    upos_labels_masked = upos_labels[upos_mask]
    upos_probs_masked  = upos_probs[upos_mask]

    upos_acc = (upos_preds_masked == upos_labels_masked).mean()

    # ---------------- CARREGAR JSON ----------------
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            all_data = json.load(f)
    else:
        all_data = {}

    # ---------------- META ----------------
    if "META" not in all_data:
        all_data["META"] = {
            "model_name": model_name,
            "pretrained_model": PRETRAINED_MODEL,
            "fold": FOLD,
            "trial": TRIAL,
            "run_id": run_id
        }

    # ---------------- STORAGE ----------------
    if "RANK_PREDICTIONS" not in all_data:
        all_data["RANK_PREDICTIONS"] = {
            "deprel": [],
            "upos": [],
            "head": []
        }

    def get_correct_rank(prob_vector, correct_label):
        sorted_indices = np.argsort(prob_vector)[::-1]
        return int(np.where(sorted_indices == correct_label)[0][0] + 1)

    # ---------------- DEPREL ----------------
    for i in range(len(deprel_preds_masked)):
        probs = deprel_probs_masked[i]
        correct_label = int(deprel_labels_masked[i])
        pred_label = int(deprel_preds_masked[i])

        all_data["RANK_PREDICTIONS"]["deprel"].append({
            "run_id": run_id,
            "model": model_name,
            "fold": FOLD,
            "trial": TRIAL,
            "index": i,
            "correct_label": correct_label,
            "pred_label": pred_label,
            "correct_prob": float(probs[correct_label]),
            "pred_prob": float(probs[pred_label]),
            "correct_rank": get_correct_rank(probs, correct_label)
        })

    # ---------------- UPOS ----------------
    for i in range(len(upos_preds_masked)):
        probs = upos_probs_masked[i]
        correct_label = int(upos_labels_masked[i])
        pred_label = int(upos_preds_masked[i])

        all_data["RANK_PREDICTIONS"]["upos"].append({
            "run_id": run_id,
            "model": model_name,
            "fold": FOLD,
            "trial": TRIAL,
            "index": i,
            "correct_label": correct_label,
            "pred_label": pred_label,
            "correct_prob": float(probs[correct_label]),
            "pred_prob": float(probs[pred_label]),
            "correct_rank": get_correct_rank(probs, correct_label)
        })

    # ---------------- HEAD ----------------
    for i in range(len(head_preds_masked)):
        probs = head_probs_masked[i]
        correct_label = int(head_labels_masked[i])
        pred_label = int(head_preds_masked[i])

        all_data["RANK_PREDICTIONS"]["head"].append({
            "run_id": run_id,
            "model": model_name,
            "fold": FOLD,
            "trial": TRIAL,
            "index": i,
            "correct_label": correct_label,
            "pred_label": pred_label,
            "correct_prob": float(probs[correct_label]),
            "pred_prob": float(probs[pred_label]),
            "correct_rank": get_correct_rank(probs, correct_label)
        })

    # ---------------- SALVAR ----------------
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(all_data, f, indent=2, ensure_ascii=False)

    return {
        "uas": float(uas),
        "las": float(las),
        "upos_accuracy": float(upos_acc),
    }"""

'import numpy as np\nimport json\nimport os\n\ndef softmax(x, axis=-1):\n    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))\n    return e_x / e_x.sum(axis=axis, keepdims=True)\n\n\ndef compute_metrics(eval_pred, PRETRAINED_MODEL=None, FOLD=None, TRIAL=None):\n\n    save_path = "epoch_predictions/predictions_results.json"\n    os.makedirs(os.path.dirname(save_path), exist_ok=True)\n\n    # ---------------- IDENTIDADE DO EXPERIMENTO ----------------\n    model_name = PRETRAINED_MODEL.split("/")[-1] if PRETRAINED_MODEL else "unknown_model"\n    run_id = f"{model_name}_fold{FOLD}_trial{TRIAL}"\n\n    logits, labels = eval_pred\n    deprel_logits, upos_logits, head_logits = logits\n    deprel_labels, upos_labels, head_labels = labels\n\n    # ---------------- PROBABILIDADES ----------------\n    deprel_probs = softmax(deprel_logits, axis=-1)\n    upos_probs   = softmax(upos_logits, axis=-1)\n    head_probs   = softmax(head_logits, axis=-1)\n\n    # ---------------- PREDIÇÕES --------

In [31]:
from sklearn.model_selection import KFold
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

In [32]:
from datasets import concatenate_datasets



In [33]:
import torch

if torch.cuda.is_available():
    print("GPU está ativa!")
    print(f"Nome da GPU: {torch.cuda.get_device_name(0)}")
    print(f"Número de GPUs disponíveis: {torch.cuda.device_count()}")
else:
    print("GPU não está disponível ou não foi detectada.")


GPU está ativa!
Nome da GPU: NVIDIA GeForce RTX 4090
Número de GPUs disponíveis: 1


In [34]:
import os
os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"

import wandb
#wandb.login()


In [35]:
output_directory = RESULTS_DIRECTORY
evaluation_strategy = 'epoch'
gradint_accumulation_steps = 1
learning_rate=2.2933248212781904e-05,
weight_decay=0.17042771903841708,
max_grad_norm = 1
num_train_epochs = 10 #NUM_TRAIN_EPOCHS
lr_scheduler_type = 'linear'
warmup_ratio=0.42465329939827173
logging_dir = LOGGING_DIRECTORY
logging_strategy = 'epoch'
save_strategy = 'epoch'
save_total_limit = 1
#label_names = ['xpos_label', 'deprel_label', 'upos_label', 'head_label']
label_names = ['deprel_label', 'upos_label', 'head_label']
load_best_model_at_end = True
metric_for_best_model="las"
greater_is_better = True
label_smoothing_factor = 0
#report_to = 'tensorboard'
gradient_checkpointing = False
remove_unused_columns=False

In [36]:
# Setup training arguments
'''training_args = TrainingArguments(
    #output_dir= f'./ettin-decoder-150m_parser',
    eval_strategy=evaluation_strategy,
    learning_rate=learning_rate,
    num_train_epochs=10,
    weight_decay=weight_decay,
    logging_dir=logging_dir,
    label_names=label_names,
    max_grad_norm=max_grad_norm,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    logging_strategy=logging_strategy,
    save_strategy=save_strategy,
    save_total_limit=save_total_limit,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better,
    label_smoothing_factor=label_smoothing_factor,
    #report_to=report_to,
    gradient_checkpointing=gradient_checkpointing
)'''

early_stop_callback = EarlyStoppingCallback(3)

In [37]:
!pip freeze > requirements_all.txt

# Optuna

In [38]:
import optuna
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score
import os
import json
import gc
import torch
#os.environ["WANDB_DISABLED"] = "true"

from datasets import concatenate_datasets

def save_result_to_json(result_dict, filename="results.jsonl"):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")



# K Folds

def objective(trial, name_model, train_data, valid_data, tokenizer=None, data_collator=None, name="model_run"):
    # Hiperparâmetros sugeridos pelo Optuna
    #while True:
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.1, 0.3)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.3, 0.5)
    num_train_epochs = trial.suggest_int("num_train_epochs", 40, 40)
    label_smoothing_factor = 0 #trial.suggest_float("label_smoothing_factor", 0.0, 0.2)
    #config = (weight_decay, learning_rate, warmup_ratio)
    
    import os
    os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"

    import wandb
    

    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # Suponha que seus dados estejam assim:
    # data = {'train': list de exemplos, 'val': list de exemplos}

    # 1. Juntar tudo em um único data
    #full_data = concatenate_datasets([train_data, valid_data])
    full_data = concatenate_datasets([data['train'], data['val']])
    # Converter para numpy array só para facilitar a indexação
    indices = np.arange(len(full_data))
    nerdataset = POSDataset(name_model)
    for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):
        #gc.collect()

        #torch.cuda.empty_cache()

        #torch.cuda.ipc_collect()

        
        # reinstalar_pytorch_nightly()
        wandb.init(
            entity="gdlima-universidade-federal-de-pelotas",
            project="hf-optuna",
            name=f"linear_{name_model}_trial{trial.number}_fold{fold}",  # <- aqui define o nome do run
            config={
                "learning_rate": learning_rate,
                "architecture": name_model,
                "epochs": num_train_epochs,
                "weight_decay": weight_decay,
                "warmup_ratio": warmup_ratio,
                "label_smoothing_factor": label_smoothing_factor
            },
        )
        config = AutoConfig.from_pretrained(name_model)

        model = MultiTaskSentencePredictionEncoder.from_pretrained(
            name_model,
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS),
            _fast_init=False,  # garante que _init_weights roda para Biaffine/MLP
        )

        # Verificar CPU antes de mover para GPU — distingue bug de init vs CUDA corrompido
        for _name, _p in model.named_parameters():
            if not _p.requires_grad:
                continue
            if torch.isnan(_p).any() or torch.isinf(_p).any():
                raise RuntimeError(
                    f"[CPU] Peso '{_name}' é NaN/Inf antes de mover para CUDA. "
                    "Bug de inicialização — revise _init_weights."
                )

        _device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(_device)

        # Verificar CUDA: se o peso ficou NaN/Inf só após .to("cuda") → contexto corrompido
        for _name, _p in model.named_parameters():
            if not _p.requires_grad:
                continue
            if torch.isnan(_p).any() or torch.isinf(_p).any():
                raise RuntimeError(
                    f"[CUDA] Peso '{_name}' tornou-se NaN/Inf após .to(cuda). "
                    "Contexto CUDA corrompido → Kernel → Restart → Run All Cells."
                )
        
        print(f"\n===== Fold {fold + 1} / {k} =====")

        # 3. Selecionar os dados (Dataset, não list)
        train_split = full_data.select(train_idx.tolist())
        val_split = full_data.select(val_idx.tolist())
        
        # 4. Tokenizar novamente usando sua função existente
        train_data, valid_data = nerdataset.create_data(train_split, val_split)
        

        #train_data = train_data.shuffle(seed=42).select(range(400))
        #valid_data = valid_data.shuffle(seed=42).select(range(200))
        training_args = TrainingArguments(
            # output_dir por fold: evita conflito no load_best_model_at_end
            output_dir=f"./parser_{name_model.replace('/', '_')}/fold_{fold}",
            fp16=False,
            bf16=False,  # desativar até estabilizar; reativar após confirmar convergência
            eval_strategy="epoch",
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,   # TODO: migrar para warmup_steps
            max_grad_norm=1.0,  # biaffine com 3 losses somados: 1.0 é seguro
            lr_scheduler_type="linear",
            logging_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,  # necessário para EarlyStoppingCallback funcionar
            metric_for_best_model="las",
            greater_is_better=True,
            label_smoothing_factor=0.0,
            gradient_checkpointing=False, # incompatível com biaffine em alguns casos
            remove_unused_columns=False,  # obrigatório — modelo recebe labels customizadas
            label_names=["deprel_label", "upos_label", "head_label"],
            report_to="wandb",
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            dataloader_num_workers=4,
        )

        early_stop_callback = EarlyStoppingCallback(5)
        # Inicializa o Trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_data,
            eval_dataset=valid_data,
            #tokenizer=tokenizer,          # opcional
            compute_metrics=compute_metrics,
            data_collator=data_collator,  # opcional
            callbacks=[early_stop_callback]  # se você quiser usar callbacks
        )


        trainer.train()
        eval_result = trainer.evaluate()

        result_dict = {
            "name": name_model,
            "Fold": fold+1,
            "trial_number": trial.number,
            "hyperparameters": {
                "learning_rate": learning_rate,
                "num_train_epochs": num_train_epochs,
                "weight_decay": weight_decay,
                "warmup_ratio": warmup_ratio,
                "label_smoothing_factor": label_smoothing_factor
            },
            "las": eval_result["eval_las"],
        }

        save_result_to_json(result_dict)
        print(f"Fold {fold + 1} metrics:", eval_result)

In [39]:

'''def objective(trial, name_model, train_data, valid_data, tokenizer=None, data_collator=None, optuna_count=0, name="model_run"):
    # Hiperparâmetros sugeridos pelo Optuna
    forbidden_configs = {
    (0.2147035642975132, 1.7109988776595992e-05, 0.41601400434863894),
    (0.2147035642975132, 2.208391537977642e-05, 0.41601400434863894),
    (0.2147035642975132, 2.5771715479732614e-05, 0.41601400434863894),
    (0.2147035642975132, 2.3346341563131348e-05, 0.41601400434863894),
    (0.2147035642975132, 4.818415253407452e-05, 0.41601400434863894),
}
    while True:
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
        weight_decay = trial.suggest_float("weight_decay", 0.1, 0.3)
        warmup_ratio = trial.suggest_float("warmup_ratio", 0.3, 0.5)
        num_train_epochs = trial.suggest_int("num_train_epochs", 40, 40)
        config = (weight_decay, learning_rate, warmup_ratio)

        if config not in forbidden_configs:
            break  # valor válido


    import os
    os.environ["WANDB_API_KEY"] = "7499f50c2fffad3ce3e34b7b7b02152988a1742c"

    import wandb
    
    #wandb.init(project="bracis", entity="gdlima")
    run = wandb.init(
        # Set the wandb entity where your project will be logged (generally your team name).
        entity="gdlima-universidade-federal-de-pelotas",
        # Set the wandb project where this run will be logged.
        project="bracis",
        # Track hyperparameters and run metadata.
        config={
            "learning_rate": learning_rate,
            "architecture": name_model,
            "epochs": num_train_epochs,
            "weight_decay": weight_decay,
            "warmup_ratio": warmup_ratio,
        },
    )
    #kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # Suponha que seus dados estejam assim:
    # data = {'train': list de exemplos, 'val': list de exemplos}

    # 1. Juntar tudo em um único data
    #full_data = concatenate_datasets([train_data, valid_data])
    #full_data = concatenate_datasets([data['train'], data['val']])
    # Converter para numpy array só para facilitar a indexação
    #indices = np.arange(len(full_data))

    #for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):


    config = AutoConfig.from_pretrained(name_model)

    model = MultiTaskSentencePredictionEncoder.from_pretrained(
            name_model,
            config=config,
            num_deprel_labels=len(DEPREL_LABELS),
            num_upos_labels=len(UPOS_LABELS)
        ).to("cuda" if torch.cuda.is_available() else "cpu")
        
        #print(f"\n===== Fold {fold + 1} / {k} =====")

        # 3. Selecionar os dados (Dataset, não list)
        #train_split = full_data.select(train_idx.tolist())
        #val_split = full_data.select(val_idx.tolist())

        # 4. Tokenizar novamente usando sua função existente
    #train_data, valid_data = nerdataset.create_data(train_split, val_split)


        # Setup training arguments
    training_args = TrainingArguments(
            #output_dir= f'./ettin-decoder-150m_parser',
            eval_strategy=evaluation_strategy,
            learning_rate=learning_rate,
            num_train_epochs=num_train_epochs,
            weight_decay=weight_decay,
            logging_dir=logging_dir,
            label_names=label_names,
            max_grad_norm=max_grad_norm,
            lr_scheduler_type=lr_scheduler_type,
            warmup_ratio=warmup_ratio,
            logging_strategy=logging_strategy,
            save_strategy=save_strategy,
            save_total_limit=save_total_limit,
            #load_best_model_at_end=load_best_model_at_end,
            metric_for_best_model=metric_for_best_model,
            greater_is_better=greater_is_better,
            label_smoothing_factor=label_smoothing_factor,
            #report_to=report_to,
            #per_device_train_batch_size=per_device_batch_size,
            gradient_checkpointing=gradient_checkpointing,
            remove_unused_columns=remove_unused_columns,
        )

        
        # Inicializa o Trainer
    trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_data,  # Limitando o treinamento para 10 exemplos
            eval_dataset=valid_data,   # Limitando a avaliação para 10 exemplos
            #tokenizer=tokenizer,          # opcional
            compute_metrics=compute_metrics,
            data_collator=data_collator,  # opcional
            # callbacks=[early_stop_callback]  # se você quiser usar callbacks
        )


    trainer.train()
    eval_result = trainer.evaluate()
    print(eval_result)
    result_dict = {
            "name": name_model,
            "Fold": optuna_count,
            "trial_number": trial.number,
            "hyperparameters": {
                "learning_rate": learning_rate,
                "num_train_epochs": num_train_epochs,
            },
            "las": eval_result["eval_las"],
        }

    save_result_to_json(result_dict)
    print(f"Optuna {optuna_count + 1} metrics:", eval_result)
    '''

'def objective(trial, name_model, train_data, valid_data, tokenizer=None, data_collator=None, optuna_count=0, name="model_run"):\n    # Hiperparâmetros sugeridos pelo Optuna\n    forbidden_configs = {\n    (0.2147035642975132, 1.7109988776595992e-05, 0.41601400434863894),\n    (0.2147035642975132, 2.208391537977642e-05, 0.41601400434863894),\n    (0.2147035642975132, 2.5771715479732614e-05, 0.41601400434863894),\n    (0.2147035642975132, 2.3346341563131348e-05, 0.41601400434863894),\n    (0.2147035642975132, 4.818415253407452e-05, 0.41601400434863894),\n}\n    while True:\n        learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)\n        weight_decay = trial.suggest_float("weight_decay", 0.1, 0.3)\n        warmup_ratio = trial.suggest_float("warmup_ratio", 0.3, 0.5)\n        num_train_epochs = trial.suggest_int("num_train_epochs", 40, 40)\n        config = (weight_decay, learning_rate, warmup_ratio)\n\n        if config not in forbidden_configs:\n            b

In [40]:
# Crear estudio y optimizar
study = optuna.create_study(direction="maximize")



[I 2026-06-15 09:26:27,864] A new study created in memory with name: no-name-f948ef1c-e3cb-4d92-94f2-6fd7d07c7d35


In [41]:
'''#Decoder
import torch
import gc

def limpar_gpu():
    gc.collect()                                # Limpa lixo da CPU
    torch.cuda.empty_cache()                    # Libera cache da GPU
    torch.cuda.ipc_collect()                    # Coleta memória interprocessos
    torch.cuda.reset_peak_memory_stats()        # Reseta estatísticas de memória
    torch.cuda.synchronize()                    # Garante execução sincronizada

# Exemplo de uso antes do treinamento
if torch.cuda.is_available():
    limpar_gpu()'''

'#Decoder\nimport torch\nimport gc\n\ndef limpar_gpu():\n    gc.collect()                                # Limpa lixo da CPU\n    torch.cuda.empty_cache()                    # Libera cache da GPU\n    torch.cuda.ipc_collect()                    # Coleta memória interprocessos\n    torch.cuda.reset_peak_memory_stats()        # Reseta estatísticas de memória\n    torch.cuda.synchronize()                    # Garante execução sincronizada\n\n# Exemplo de uso antes do treinamento\nif torch.cuda.is_available():\n    limpar_gpu()'

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [43]:
'''models_decoder = ["maritaca-ai/sabia-7b" , "nicholasKluge/TeenyTinyLlama-460m", "TucanoBR/Tucano-630m", "https://huggingface.co/adalbertojunior/bart-base-portuguese"]


for model_name in models_decoder:
# Otimização
    study.optimize(
        lambda trial: objective(trial, MultiTaskSentencePrediction.from_pretrained(
        model_name,
        config=config,
        num_deprel_labels=len(DEPREL_LABELS),
        num_upos_labels=len(UPOS_LABELS)
    ).to(device), train_data, valid_data,  data_collator),
        n_trials=5
    )'''



'models_decoder = ["maritaca-ai/sabia-7b" , "nicholasKluge/TeenyTinyLlama-460m", "TucanoBR/Tucano-630m", "https://huggingface.co/adalbertojunior/bart-base-portuguese"]\n\n\nfor model_name in models_decoder:\n# Otimização\n    study.optimize(\n        lambda trial: objective(trial, MultiTaskSentencePrediction.from_pretrained(\n        model_name,\n        config=config,\n        num_deprel_labels=len(DEPREL_LABELS),\n        num_upos_labels=len(UPOS_LABELS)\n    ).to(device), train_data, valid_data,  data_collator),\n        n_trials=5\n    )'

In [44]:
from transformers import AutoModel, AutoConfig
from transformers import BertConfig


def build_model(model_name, num_deprel_labels, num_upos_labels):

    config = AutoConfig.from_pretrained(model_name)
    
    model = MultiTaskSentencePrediction.from_pretrained(
    model_name,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS),
    num_upos_labels=len(UPOS_LABELS)
)

    return model

In [45]:
!pip install numpy==1.24.2


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [46]:
!pip install protobuf==3.20.3

  Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (679 bytes)
Using cached protobuf-3.20.3-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (1.1 MB)


  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires protobuf!=5.28.0,!=5.29.0,<8,>4.21.0, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [47]:
!export PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python

In [48]:
!export WANDB_NOTEBOOK_NAME="optuna_models"

In [49]:
!pip install --upgrade wandb

  Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl (327 kB)


  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:


      Successfully uninstalled protobuf-3.20.3



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [50]:
from transformers import set_seed

set_seed(42)

In [51]:
import os
import random
import numpy as np
import torch

def seed_everything(seed: int = 42):
    # Python
    random.seed(seed)
    
    # Numpy
    np.random.seed(seed)
    
    # PyTorch (CPU)
    torch.manual_seed(seed)
    
    # PyTorch (GPU)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Determinismo (importante!)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Algumas libs usam isso
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    # Para transformers (às vezes ajuda)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    print(f"Seed definida como {seed}")

# Uso
seed_everything(42)

Seed definida como 42


In [52]:
#models_encoder = ["neuralmind/bert-base-portuguese-cased", "google-bert/bert-base-multilingual-cased", "neuralmind/bert-large-portuguese-cased"]
models_encoder = ["amadeusai/modernJabuticaBERT-Base-1k"]
#models_encoder = ["google-bert/bert-base-multilingual-cased"] #, "neuralmind/bert-large-portuguese-cased"]
#models_encoder = ["google-bert/bert-base-multilingual-cased", "neuralmind/bert-large-portuguese-cased" , "distilbert/distilbert-base-uncased", "google-bert/bert-large-uncased"]

optuna_count = 0
#print(len(train_data))

#train_data = train_data.shuffle(seed=42).select(range(1000))
#valid_data = valid_data.shuffle(seed=42).select(range(1000))
for model_name in models_encoder:   

    study = optuna.create_study(direction="maximize")

    print("="*50)
    print(f'Modelo: {model_name}')
    print("="*50)

    study.optimize(
        lambda trial: objective(
            trial,
            model_name,
            train_data,
            valid_data,
            data_collator,
            #optuna_count=optuna_count  # Passando o valor atual de optuna_count
        ),
        n_trials=10
    )

    # Incrementar optuna_count após a execução de cada trial
    optuna_count += 1

[I 2026-06-15 09:26:31,715] A new study created in memory with name: no-name-71851516-8f8b-4d1e-a0a5-fa484b8b25e7


Modelo: amadeusai/modernJabuticaBERT-Base-1k


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: gdlima (gdlima-universidade-federal-de-pelotas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.27.2


...

Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  1.02it/s]

Training Loss,Validation Loss,Epoch,Uas,Las,Upos Accuracy
0.215483,1.494592,34,0.692502,0.670494,0.977011


[W 2026-06-17 10:56:25,753] Trial 9 failed with parameters: {'learning_rate': 1.3162609676636067e-05, 'weight_decay': 0.1633941432427773, 'warmup_ratio': 0.35628848630322457, 'num_train_epochs': 40} because of the following error: The value None could not be cast to float..


[W 2026-06-17 10:56:25,755] Trial 9 failed with value None.


Fold 5 metrics: {'eval_loss': 1.4945924282073975, 'eval_uas': 0.692502193095619, 'eval_las': 0.670493833531142, 'eval_upos_accuracy': 0.9770111976882192}


In [53]:
#print("Melhor Modelo:", study.best_value)
#print("Melhor Hiperparametros:", study.best_params)

# K-Folds

In [54]:
'''from sklearn.model_selection import KFold
import numpy as np
from datasets import concatenate_datasets
# Suponha que seus dados estejam assim:
# data = {'train': list de exemplos, 'val': list de exemplos}

# 1. Juntar tudo em um único data
full_data = concatenate_datasets([data['train'], data['val']])
# 2. Criar os índices para K-Fold
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

# Converter para numpy array só para facilitar a indexação
indices = np.arange(len(full_data))

for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):
    print(f"\n===== Fold {fold + 1} / {k} =====")

    # 3. Selecionar os dados (Dataset, não list)
    train_split = full_data.select(train_idx.tolist())
    val_split = full_data.select(val_idx.tolist())

    # 4. Tokenizar novamente usando sua função existente
    train_data, valid_data = nerdataset.create_data(train_split, val_split)

    # 5. (Re)criar o modelo (importante para que cada fold comece do zero)
    #model = model_init()  # define essa função para criar um novo modelo
    model = MultiTaskSentencePrediction.from_pretrained(
    PRETRAINED_MODEL_NAME,
    config=config,
    num_deprel_labels=len(DEPREL_LABELS), num_upos_labels=len(UPOS_LABELS), num_head_labels=100
    )
    # 6. Criar o Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=valid_data,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        # tokenizer=tokenizer,  # se necessário
        # callbacks=[early_stop_callback]  # se estiver usando
    )

    # 7. Treinar
    trainer.train()

    # 8. Avaliar
    metrics = trainer.evaluate()
    print(f"Fold {fold + 1} metrics:", metrics)
'''

'from sklearn.model_selection import KFold\nimport numpy as np\nfrom datasets import concatenate_datasets\n# Suponha que seus dados estejam assim:\n# data = {\'train\': list de exemplos, \'val\': list de exemplos}\n\n# 1. Juntar tudo em um único data\nfull_data = concatenate_datasets([data[\'train\'], data[\'val\']])\n# 2. Criar os índices para K-Fold\nk = 5\nkf = KFold(n_splits=k, shuffle=True, random_state=42)\n\n# Converter para numpy array só para facilitar a indexação\nindices = np.arange(len(full_data))\n\nfor fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):\n    print(f"\n===== Fold {fold + 1} / {k} =====")\n\n    # 3. Selecionar os dados (Dataset, não list)\n    train_split = full_data.select(train_idx.tolist())\n    val_split = full_data.select(val_idx.tolist())\n\n    # 4. Tokenizar novamente usando sua função existente\n    train_data, valid_data = nerdataset.create_data(train_split, val_split)\n\n    # 5. (Re)criar o modelo (importante para que cada fold comece

# Treinamento Modelo

In [55]:
# Setup training arguments
"""training_args = TrainingArguments(
    #output_dir= f'./ettin-decoder-150m_parser',
    eval_strategy=evaluation_strategy,
    learning_rate=4.350420087020882e-05,
    num_train_epochs=40,
    weight_decay=0.23448065843551166,
    logging_dir=logging_dir,
    label_names=label_names,
    max_grad_norm=max_grad_norm,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=0.4962097355105568,
    logging_strategy=logging_strategy,
    save_strategy=save_strategy,
    save_total_limit=save_total_limit,
    load_best_model_at_end=load_best_model_at_end,
    metric_for_best_model=metric_for_best_model,
    greater_is_better=greater_is_better,
    label_smoothing_factor=label_smoothing_factor,
    #report_to=report_to,
    gradient_checkpointing=gradient_checkpointing
)

#early_stop_callback = EarlyStoppingCallback(3)"""

"training_args = TrainingArguments(\n    #output_dir= f'./ettin-decoder-150m_parser',\n    eval_strategy=evaluation_strategy,\n    learning_rate=4.350420087020882e-05,\n    num_train_epochs=40,\n    weight_decay=0.23448065843551166,\n    logging_dir=logging_dir,\n    label_names=label_names,\n    max_grad_norm=max_grad_norm,\n    lr_scheduler_type=lr_scheduler_type,\n    warmup_ratio=0.4962097355105568,\n    logging_strategy=logging_strategy,\n    save_strategy=save_strategy,\n    save_total_limit=save_total_limit,\n    load_best_model_at_end=load_best_model_at_end,\n    metric_for_best_model=metric_for_best_model,\n    greater_is_better=greater_is_better,\n    label_smoothing_factor=label_smoothing_factor,\n    #report_to=report_to,\n    gradient_checkpointing=gradient_checkpointing\n)\n\n#early_stop_callback = EarlyStoppingCallback(3)"

In [56]:
"""run = wandb.init(
        # Set the wandb entity where your project will be logged (generally your team name).
        entity="gdlima-universidade-federal-de-pelotas",
        name=f"modernJabutica-Selecionado",  # <- aqui define o nome do run
        # Set the wandb project where this run will be logged.
        project="hf-optuna",
        # Track hyperparameters and run metadata.
        config={
            "learning_rate": 4.350420087020882e-05,
            "architecture": "modernJabutica-Selecionado",
            "epochs": 40,
            "weight_decay": 0.23448065843551166,
            "warmup_ratio": 0.4962097355105568,
        },
    )"""

'run = wandb.init(\n        # Set the wandb entity where your project will be logged (generally your team name).\n        entity="gdlima-universidade-federal-de-pelotas",\n        name=f"modernJabutica-Selecionado",  # <- aqui define o nome do run\n        # Set the wandb project where this run will be logged.\n        project="hf-optuna",\n        # Track hyperparameters and run metadata.\n        config={\n            "learning_rate": 4.350420087020882e-05,\n            "architecture": "modernJabutica-Selecionado",\n            "epochs": 40,\n            "weight_decay": 0.23448065843551166,\n            "warmup_ratio": 0.4962097355105568,\n        },\n    )'

In [57]:
#model

In [58]:
"""# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    
    #callbacks=[early_stop_callback]
)"""


'# Initialize the Trainer\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_data,\n    eval_dataset=valid_data,\n    # tokenizer=tokenizer,\n    compute_metrics=compute_metrics,\n    data_collator=data_collator,\n    \n    #callbacks=[early_stop_callback]\n)'

In [59]:
#trainer.train()


In [60]:
#trainer.evaluate()

In [61]:
"""import json
import pandas as pd
from pathlib import Path

# ======================================================
# CAMINHO DO ARQUIVO JSONL
# (1 JSON por linha)
# ======================================================

json_file = "results_bertimbau_large.jsonl"

# ======================================================
# LEITURA DO ARQUIVO
# ======================================================

records = []

with open(json_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        # ignora linhas vazias
        if not line:
            continue

        obj = json.loads(line)

        row = {
            "model": obj["name"],
            "fold": obj["Fold"],
            "trial": obj["trial_number"],
            "las": obj["las"],
        }

        # adiciona hiperparâmetros
        row.update(obj["hyperparameters"])

        records.append(row)

# ======================================================
# DATAFRAME
# ======================================================

df = pd.DataFrame(records)

print("\n================ DADOS CARREGADOS ================\n")
print(df.head())

# ======================================================
# CROSS VALIDATION
# ======================================================

group_cols = [
    "model",
    "trial",
    "learning_rate",
    "num_train_epochs",
    "weight_decay",
    "warmup_ratio",
]

cv_results = (
    df.groupby(group_cols)
    .agg(
        mean_las=("las", "mean"),
        std_las=("las", "std"),
        min_las=("las", "min"),
        max_las=("las", "max"),
        folds=("las", "count"),
    )
    .reset_index()
)

# ordena pelo melhor LAS médio
cv_results = cv_results.sort_values(
    by="mean_las",
    ascending=False
)

# ======================================================
# MELHOR HIPERPARÂMETRO
# ======================================================

best = cv_results.iloc[0]

print("\n================ MELHOR HIPERPARÂMETRO ================\n")

print(f"Modelo: {best['model']}")
print(f"Trial: {best['trial']}")

print("\nHiperparâmetros:")
print(f"  learning_rate    = {best['learning_rate']}")
print(f"  num_train_epochs = {best['num_train_epochs']}")
print(f"  weight_decay     = {best['weight_decay']}")
print(f"  warmup_ratio     = {best['warmup_ratio']}")

print("\nResultados Cross Validation:")
print(f"  Mean LAS = {best['mean_las']:.6f}")
print(f"  Std LAS  = {best['std_las']:.6f}")
print(f"  Min LAS  = {best['min_las']:.6f}")
print(f"  Max LAS  = {best['max_las']:.6f}")
print(f"  Folds    = {best['folds']}")

# ======================================================
# SALVAR RANKING COMPLETO
# ======================================================

output_csv = "cv_results.csv"

cv_results.to_csv(output_csv, index=False)

print(f"\nRanking salvo em: {output_csv}")

# ======================================================
# TOP 10
# ======================================================

print("\n================ TOP 10 ================\n")

print(
    cv_results[
        [
            "model",
            "trial",
            "mean_las",
            "std_las",
            "learning_rate",
            "weight_decay",
            "warmup_ratio",
        ]
    ]
    .head(10)
    .to_string(index=False)
)"""

'import json\nimport pandas as pd\nfrom pathlib import Path\n\n# ======================================================\n# CAMINHO DO ARQUIVO JSONL\n# (1 JSON por linha)\n# ======================================================\n\njson_file = "results_bertimbau_large.jsonl"\n\n# ======================================================\n# LEITURA DO ARQUIVO\n# ======================================================\n\nrecords = []\n\nwith open(json_file, "r", encoding="utf-8") as f:\n    for line in f:\n        line = line.strip()\n\n        # ignora linhas vazias\n        if not line:\n            continue\n\n        obj = json.loads(line)\n\n        row = {\n            "model": obj["name"],\n            "fold": obj["Fold"],\n            "trial": obj["trial_number"],\n            "las": obj["las"],\n        }\n\n        # adiciona hiperparâmetros\n        row.update(obj["hyperparameters"])\n\n        records.append(row)\n\n# ======================================================\n# DATAF